# SupplyVision

## Notebook 01 - Dataset Audit

### Objective

Understand the structure, quality, and relationships of the retail data warehouse dataset before any transformation or analysis.

### Business Context

Retail organizations receive data from multiple operational systems.
Before building dashboards or reports, analysts must verify the quality,
structure, and relationships of incoming datasets.

### Tasks

- Load every CSV
- Inspect dimensions
- Check data types
- Identify missing values
- Check duplicates
- Identify candidate Primary Keys
- Identify candidate Foreign Keys
- Produce dataset summary

### Expected Output

A complete audit report describing every table and identifying
potential data quality issues before ETL begins.

In [2]:
import pandas as pd
from pathlib import Path

#Display settings
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows",20)

In [3]:
DATA_PATH = Path("../data/raw")

In [4]:
csv_files = list(DATA_PATH.glob("*.csv"))
csv_files

[WindowsPath('../data/raw/categories.csv'),
 WindowsPath('../data/raw/customers.csv'),
 WindowsPath('../data/raw/employees.csv'),
 WindowsPath('../data/raw/orders.csv'),
 WindowsPath('../data/raw/order_items.csv'),
 WindowsPath('../data/raw/payments.csv'),
 WindowsPath('../data/raw/products.csv'),
 WindowsPath('../data/raw/promotions.csv'),
 WindowsPath('../data/raw/returns.csv'),
 WindowsPath('../data/raw/shipments.csv'),
 WindowsPath('../data/raw/stores.csv'),
 WindowsPath('../data/raw/suppliers.csv')]

In [5]:
tables = {}

for file in csv_files:
    tables[file.stem] = pd.read_csv(file)

print(f"Loaded{len(tables)}tables.")

Loaded12tables.


In [6]:
summary =[]

for name,df in tables.items():
    summary.append({
        "Table" : name,
        "Rows" : df.shape[0],
        "Columns" : df.shape[1],
        "Missing Values" : int(df.isnull().sum().sum()),
        "Duplicate Rows" : int(df.duplicated().sum())
    })

summary_df = pd.DataFrame(summary)

summary_df

,Table,Rows,Columns,Missing Values,Duplicate Rows
0,categories,30,2,0,0
1,customers,50000,3,0,0
2,employees,1000,3,0,0
3,orders,300000,5,0,0
4,order_items,600000,5,0,0
5,payments,300000,3,0,0
6,products,10000,4,0,0
7,promotions,50,2,0,0
8,returns,30000,3,0,0
9,shipments,300000,3,0,0


In [7]:
for name, df in tables.items():
    print("=" * 70)
    print(f"TABLE: {name.upper()}")
    print("=" * 70)
    print(df.dtypes)
    print()

TABLE: CATEGORIES
category_id      int64
category_name      str
dtype: object

TABLE: CUSTOMERS
customer_id    int64
city             str
signup_date      str
dtype: object

TABLE: EMPLOYEES
employee_id    int64
store_id       int64
salary         int64
dtype: object

TABLE: ORDERS
order_id        int64
customer_id     int64
store_id        int64
order_date        str
promotion_id    int64
dtype: object

TABLE: ORDER_ITEMS
order_item_id    int64
order_id         int64
product_id       int64
qty              int64
price            int64
dtype: object

TABLE: PAYMENTS
payment_id    int64
order_id      int64
amount        int64
dtype: object

TABLE: PRODUCTS
product_id     int64
category_id    int64
supplier_id    int64
price          int64
dtype: object

TABLE: PROMOTIONS
promotion_id    int64
discount        int64
dtype: object

TABLE: RETURNS
return_id        int64
order_item_id    int64
refund           int64
dtype: object

TABLE: SHIPMENTS
shipment_id    int64
order_id       int64
st

In [8]:
for name, df in tables.items():
    print("=" * 70)
    print(name.upper())
    print("=" * 70)
    display(df.head(3))

CATEGORIES


,category_id,category_name
0,1,Cat_1
1,2,Cat_2
2,3,Cat_3


CUSTOMERS


,customer_id,city,signup_date
0,1,Mumbai,2021-02-16
1,2,Bangalore,2019-06-22
2,3,Pune,2022-01-25


EMPLOYEES


...

,shipment_id,order_id,status
0,1,1,delivered
1,2,2,shipped
2,3,3,late


STORES


,store_id,city
0,1,Pune
1,2,Pune
2,3,Delhi


SUPPLIERS


,supplier_id,country
0,1,India
1,2,India
2,3,India


## Candidate Primary Keys

In [9]:
for table_name, df in tables.items():

    print("=" * 70)
    print(f"{table_name.upper()}")

    for column in df.columns:

        if df[column].is_unique:
            print(f"✅ {column} is UNIQUE")

CATEGORIES
✅ category_id is UNIQUE
✅ category_name is UNIQUE
CUSTOMERS
✅ customer_id is UNIQUE
EMPLOYEES
✅ employee_id is UNIQUE
ORDERS
✅ order_id is UNIQUE
ORDER_ITEMS
✅ order_item_id is UNIQUE
PAYMENTS
✅ payment_id is UNIQUE
✅ order_id is UNIQUE
PRODUCTS
✅ product_id is UNIQUE
PROMOTIONS
✅ promotion_id is UNIQUE
RETURNS
✅ return_id is UNIQUE
SHIPMENTS
✅ shipment_id is UNIQUE
✅ order_id is UNIQUE
STORES
✅ store_id is UNIQUE
SUPPLIERS
✅ supplier_id is UNIQUE


## Foreign Key Integrity Validation

In [11]:
fk_checks = {
    "orders.customer_id -> customers.customer_id":
        tables["orders"]["customer_id"].isin(
            tables["customers"]["customer_id"]
        ).all(),

    "orders.store_id -> stores.store_id":
        tables["orders"]["store_id"].isin(
            tables["stores"]["store_id"]
        ).all(),

    "orders.promotion_id -> promotions.promotion_id":
        tables["orders"]["promotion_id"].isin(
            tables["promotions"]["promotion_id"]
        ).all(),

    "order_items.order_id -> orders.order_id":
        tables["order_items"]["order_id"].isin(
            tables["orders"]["order_id"]
        ).all(),

    "order_items.product_id -> products.product_id":
        tables["order_items"]["product_id"].isin(
            tables["products"]["product_id"]
        ).all(),

    "products.category_id -> categories.category_id":
        tables["products"]["category_id"].isin(
            tables["categories"]["category_id"]
        ).all(),

    "products.supplier_id -> suppliers.supplier_id":
        tables["products"]["supplier_id"].isin(
            tables["suppliers"]["supplier_id"]
        ).all(),

    "payments.order_id -> orders.order_id":
        tables["payments"]["order_id"].isin(
            tables["orders"]["order_id"]
        ).all(),

    "shipments.order_id -> orders.order_id":
        tables["shipments"]["order_id"].isin(
            tables["orders"]["order_id"]
        ).all(),

    "returns.order_item_id -> order_items.order_item_id":
        tables["returns"]["order_item_id"].isin(
            tables["order_items"]["order_item_id"]
        ).all(),
}

for relation, status in fk_checks.items():
    print(f"{'✅' if status else '❌'} {relation}")

✅ orders.customer_id -> customers.customer_id
✅ orders.store_id -> stores.store_id
✅ orders.promotion_id -> promotions.promotion_id
✅ order_items.order_id -> orders.order_id
✅ order_items.product_id -> products.product_id
✅ products.category_id -> categories.category_id
✅ products.supplier_id -> suppliers.supplier_id
✅ payments.order_id -> orders.order_id
✅ shipments.order_id -> orders.order_id
✅ returns.order_item_id -> order_items.order_item_id
